In [ ]:
# ============================== 0. Install ==============================
import subprocess, sys, os
def _pip(*pk):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
try:
    import yfinance  # noqa
except ImportError:
    _pip("yfinance")
try:
    import docx  # noqa
except ImportError:
    _pip("python-docx")
try:
    import statsmodels  # noqa
except ImportError:
    _pip("statsmodels")


In [ ]:
# ============================== 1. Configuration ==============================
MODE = os.environ.get("SIGMA_MODE", "FULL")        # "FULL" for the paper, "QUICK" for a dry run
USE_DRIVE = True                                     # keep cache + outputs on Google Drive (recommended for long runs)
SYNTHETIC = os.environ.get("SIGMA_SYNTHETIC", "0") == "1"   # offline self-test only (never for the paper)

START, END = "2010-06-29", "2025-10-11"             # END is exclusive in yfinance -> last day 2025-10-10

TICKERS = {  # symbol: (name, sector / type, country, price unit)
    "TSLA":        ("Tesla, Inc.",               "Consumer discretionary (automotive)", "USA",   "USD"),
    "AMZN":        ("Amazon.com, Inc.",          "Consumer discretionary (e-commerce)", "USA",   "USD"),
    "GOOG":        ("Alphabet Inc. (Class C)",   "Communication services",             "USA",   "USD"),
    "NVDA":        ("NVIDIA Corporation",        "Information technology",             "USA",   "USD"),
    "MSFT":        ("Microsoft Corporation",     "Information technology",             "USA",   "USD"),
    "JPM":         ("JPMorgan Chase & Co.",      "Financials",                         "USA",   "USD"),
    "XOM":         ("Exxon Mobil Corporation",   "Energy",                             "USA",   "USD"),
    "JNJ":         ("Johnson & Johnson",         "Health care",                        "USA",   "USD"),
    "^GSPC":       ("S&P 500 index",             "Broad large-cap index",              "USA",   "points"),
    "IWM":         ("iShares Russell 2000 ETF",  "Small-cap equity fund",              "USA",   "USD"),
    "^NSEI":       ("NIFTY 50 index",            "Broad large-cap index",              "India", "points"),
    "RELIANCE.NS": ("Reliance Industries Ltd.",  "Energy and conglomerate",            "India", "INR"),
}
CORE = ["TSLA", "AMZN", "GOOG", "NVDA", "MSFT"]      # the five series of the original submission

TRAIN_FRAC, VAL_FRAC = 0.70, 0.10                    # test = last 20 %
N_WF_FOLDS = 4
ARIMA_REFIT_EVERY = 20
COST_BPS = 10                                         # one-way transaction cost + slippage, basis points
VOL_WINDOW = 21

if MODE == "FULL":
    SEEDS_MAIN, SEEDS_WF, SEEDS_SENS = [0, 1, 2, 3, 4], [0, 1, 2], [0]
    MAX_EPOCHS, PATIENCE = 200, 20
    ARIMA_P, ARIMA_Q = range(0, 6), range(0, 3)
else:
    SEEDS_MAIN, SEEDS_WF, SEEDS_SENS = [0, 1], [0], [0]
    MAX_EPOCHS, PATIENCE = 15, 4
    ARIMA_P, ARIMA_Q = range(0, 3), range(0, 2)

if os.environ.get("SIGMA_TICKERS"):                  # used only by the offline self-test
    _keep = os.environ["SIGMA_TICKERS"].split(",")
    TICKERS = {k: v for k, v in TICKERS.items() if k in _keep}
    CORE = [c for c in CORE if c in TICKERS]
if os.environ.get("SIGMA_EPOCHS"):
    MAX_EPOCHS = int(os.environ["SIGMA_EPOCHS"]); PATIENCE = 2

from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/SIGMA_Stock_revision")
elif IN_COLAB:
    BASE = Path("/content/SIGMA_Stock_revision")
else:
    BASE = Path(os.environ.get("SIGMA_BASE", "./SIGMA_Stock_revision"))
DATA_DIR, RUN_DIR, FIG_DIR, TAB_DIR, DOC_DIR = [BASE / d for d in ("data", "runs_" + MODE, "figures", "tables", "manuscript")]
for d in (DATA_DIR, RUN_DIR, FIG_DIR, TAB_DIR, DOC_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Mode:", MODE, "| outputs in", BASE)


Mounted at /content/drive
Mode: FULL | outputs in /content/drive/MyDrive/SIGMA_Stock_revision


In [ ]:
# ============================== 2. Imports ==============================
import json, time, math, warnings, platform, datetime as dt, hashlib
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg") if not IN_COLAB else None
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow import keras
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt
import statsmodels

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
                     "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "legend.fontsize": 8,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.alpha": 0.3})
C_ACT, C_LSTM, C_PER, C_ARIMA, C_HOLT = "#222222", "#1f5fbf", "#d4722b", "#2a9d5b", "#8e44ad"


In [ ]:
# ============================== 3. Data ==============================
def _safe(t): return t.replace("^", "IDX_").replace(".", "_")

def _synthetic(t):
    """Offline self-test only: geometric random walk on business days."""
    rng = np.random.default_rng(int(hashlib.md5(t.encode()).hexdigest(), 16) % 2**32)
    idx = pd.bdate_range(START, "2025-10-10")
    r = rng.normal(0.0005, 0.02, len(idx))
    p = 20 * np.exp(np.cumsum(r))
    return pd.DataFrame({"Open": p, "High": p * 1.01, "Low": p * 0.99, "Close": p, "Volume": 1e6}, index=idx)

def load_prices(t):
    f = DATA_DIR / f"{_safe(t)}.csv"
    if f.exists():
        df = pd.read_csv(f, index_col=0, parse_dates=True)
    else:
        if SYNTHETIC:
            df = _synthetic(t)
        else:
            import yfinance as yf
            df = yf.Ticker(t).history(start=START, end=END, auto_adjust=True, actions=False)
            if df.empty:
                raise RuntimeError(f"No data returned for {t}")
            df.index = df.index.tz_localize(None).normalize()
            df = df[["Open", "High", "Low", "Close", "Volume"]]
        df.to_csv(f)
    return df

meta_file = DATA_DIR / "download_info.json"
if not meta_file.exists():
    import yfinance as yf
    json.dump({"download_date_utc": dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M"),
               "yfinance_version": getattr(yf, "__version__", "n/a"),
               "settings": "Ticker.history(start=%s, end=%s, auto_adjust=True, actions=False)" % (START, END),
               "synthetic": SYNTHETIC}, open(meta_file, "w"), indent=2)
DL_INFO = json.load(open(meta_file))

SERIES, RAW, QUALITY = {}, {}, []
for t in TICKERS:
    df = load_prices(t)
    RAW[t] = df
    n_raw = len(df)
    dup = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated()]
    n_nan = int(df["Open"].isna().sum())
    n_nonpos = int((df["Open"] <= 0).sum())
    s = df["Open"][(df["Open"] > 0) & df["Open"].notna()].astype(float)   # no imputation: keep real trading days only
    SERIES[t] = s
    n = len(s)
    i_val, i_test = int(TRAIN_FRAC * n), int((TRAIN_FRAC + VAL_FRAC) * n)
    zero_moves = float((np.diff(s.values) == 0).mean() * 100)
    QUALITY.append(dict(ticker=t, name=TICKERS[t][0], sector=TICKERS[t][1], country=TICKERS[t][2], unit=TICKERS[t][3],
                        first=s.index[0].date(), last=s.index[-1].date(), n_raw=n_raw, n_missing=n_nan + n_nonpos + dup,
                        n_used=n, zero_move_pct=zero_moves,
                        train=f"{s.index[0].date()} – {s.index[i_val-1].date()}",
                        val=f"{s.index[i_val].date()} – {s.index[i_test-1].date()}",
                        test=f"{s.index[i_test].date()} – {s.index[-1].date()}",
                        n_train=i_val, n_val=i_test - i_val, n_test=n - i_test))
QUALITY = pd.DataFrame(QUALITY)
QUALITY.to_csv(TAB_DIR / "data_quality.csv", index=False)
print(QUALITY[["ticker", "first", "last", "n_used", "n_missing", "n_train", "n_val", "n_test"]].to_string(index=False))

def split_points(n):
    return int(TRAIN_FRAC * n), int((TRAIN_FRAC + VAL_FRAC) * n)


     ticker      first       last  n_used  n_missing  n_train  n_val  n_test
       TSLA 2010-06-29 2025-10-10    3846          0     2692    384     770
       AMZN 2010-06-29 2025-10-10    3846          0     2692    384     770
       GOOG 2010-06-29 2025-10-10    3846          0     2692    384     770
       NVDA 2010-06-29 2025-10-10    3846          0     2692    384     770
       MSFT 2010-06-29 2025-10-10    3846          0     2692    384     770
        JPM 2010-06-29 2025-10-10    3846          0     2692    384     770
        XOM 2010-06-29 2025-10-10    3846          0     2692    384     770
        JNJ 2010-06-29 2025-10-10    3846          0     2692    384     770
      ^GSPC 2010-06-29 2025-10-10    3846          0     2692    384     770
        IWM 2010-06-29 2025-10-10    3846          0     2692    384     770
      ^NSEI 2010-06-29 2025-10-10    3754          0     2627    376     751
RELIANCE.NS 2010-06-29 2025-10-10    3774          0     2641    378     755

In [ ]:
# ============================== 4. Metrics and statistical tests ==============================
def price_metrics(y, yhat, yprev, naive_scale, rmse_persist=None):
    e = y - yhat
    rmse = float(np.sqrt(np.mean(e ** 2)))
    out = dict(RMSE=rmse, MAE=float(np.mean(np.abs(e))), MAPE=float(np.mean(np.abs(e) / np.abs(y)) * 100),
               R2=float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)),
               nRMSE=float(rmse / np.mean(y) * 100), MASE=float(np.mean(np.abs(e)) / naive_scale))
    if rmse_persist is not None:
        out["U2"] = rmse / rmse_persist
    r, rh = y / yprev - 1, yhat / yprev - 1
    out["retRMSE"] = float(np.sqrt(np.mean((r - rh) ** 2)) * 100)
    out["R2oos"] = float((1 - np.sum((r - rh) ** 2) / np.sum(r ** 2)) * 100)
    m = (r != 0) & (rh != 0)
    out["DA"] = float(np.mean(np.sign(r[m]) == np.sign(rh[m])) * 100) if m.sum() > 0 else np.nan
    out["up_share"] = float(np.mean(rh[m] > 0) * 100) if m.sum() > 0 else np.nan
    out["PT_p"] = pesaran_timmermann(r[m], rh[m]) if m.sum() > 10 else np.nan
    return out

def pesaran_timmermann(r, rh):
    """One-sided Pesaran–Timmermann (1992) test of directional predictability."""
    n = len(r); x, z = (rh > 0).astype(float), (r > 0).astype(float)
    P = np.mean(x == z); py, px = z.mean(), x.mean()
    Ps = py * px + (1 - py) * (1 - px)
    vP = Ps * (1 - Ps) / n
    vPs = ((2 * py - 1) ** 2 * px * (1 - px) / n + (2 * px - 1) ** 2 * py * (1 - py) / n
           + 4 * py * px * (1 - py) * (1 - px) / n ** 2)
    v = vP - vPs
    if v <= 1e-12:
        return np.nan               # all calls in one direction: test undefined
    return float(1 - stats.norm.cdf((P - Ps) / np.sqrt(v)))

def dm_test(e1, e2, h=1):
    """Diebold–Mariano test with Harvey–Leybourne–Newbold correction, squared-error loss.
    Negative statistic -> model 1 (e1) more accurate. Returns (stat, two-sided p)."""
    d = e1 ** 2 - e2 ** 2
    n = len(d); dbar = d.mean()
    gam = [np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n for k in range(h)]
    var = (gam[0] + 2 * sum(gam[1:])) / n
    if var <= 0:
        return np.nan, np.nan
    dm = dbar / np.sqrt(var)
    dm *= np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    p = 2 * (1 - stats.t.cdf(abs(dm), df=n - 1))
    return float(dm), float(p)

def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den; w = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - w, c + w

def trading_stats(r, pos, cost_bps):
    """Long/flat: pos[j]=1 means hold from open j-1 to open j. Cost charged on every position change."""
    ch = np.abs(np.diff(np.concatenate([[0.0], pos])))
    net = pos * r - ch * cost_bps / 1e4
    eq = np.cumprod(1 + net)
    yrs = len(r) / 252
    ann = eq[-1] ** (1 / yrs) - 1
    vol = net.std() * np.sqrt(252)
    sharpe = net.mean() / net.std() * np.sqrt(252) if net.std() > 0 else np.nan
    mdd = float(np.max(1 - eq / np.maximum.accumulate(eq)))
    return dict(ann_ret=ann * 100, ann_vol=vol * 100, sharpe=sharpe, mdd=mdd * 100, trades=int(ch.sum()), equity=eq)

def naive_scale(y, end):
    return float(np.mean(np.abs(np.diff(y[:end]))))


In [ ]:
# ============================== 5. Benchmark forecasts (one-step, rolling origin) ==============================
def bench_persistence(y, idx): return y[idx - 1]
def bench_drift(y, idx): return y[idx - 1] + (y[idx - 1] - y[0]) / (idx - 1)
def bench_snaive(y, idx, m=5): return y[idx - m]

def bench_ets(y, fit_end, idx, trend):
    yf_ = y[:fit_end]
    m = (Holt(yf_, initialization_method="estimated").fit() if trend
         else SimpleExpSmoothing(yf_, initialization_method="estimated").fit())
    a = m.params["smoothing_level"]; b = m.params.get("smoothing_trend", 0.0) if trend else 0.0
    l = m.params["initial_level"]; tr = m.params.get("initial_trend", 0.0) if trend else 0.0
    b = 0.0 if b is None or (isinstance(b, float) and np.isnan(b)) else b
    tr = 0.0 if tr is None or (isinstance(tr, float) and np.isnan(tr)) else tr
    f = np.empty(len(y))
    for t in range(len(y)):
        f[t] = l + tr
        ln = a * y[t] + (1 - a) * (l + tr)
        tr = b * (ln - l) + (1 - b) * tr
        l = ln
    return f[idx], dict(alpha=a, beta=b)

def bench_linreg(y, fit_end, idx, L=30):
    tr = np.arange(L, fit_end)
    X = np.stack([y[j - L:j] for j in tr]); lr = LinearRegression().fit(X, y[tr])
    return lr.predict(np.stack([y[j - L:j] for j in idx]))

def arima_select(yfit):
    adf_lvl = adfuller(yfit, autolag="AIC")[1]
    adf_dif = adfuller(np.diff(yfit), autolag="AIC")[1]
    d = 0 if adf_lvl < 0.05 else (1 if adf_dif < 0.05 else 2)
    trends = ["c"] if d == 0 else (["n", "t"] if d == 1 else ["n"])
    best = None
    for p in ARIMA_P:
        for q in ARIMA_Q:
            for tr in trends:
                try:
                    res = ARIMA(yfit, order=(p, d, q), trend=tr).fit()
                except Exception:
                    continue
                if np.isfinite(res.aic) and (best is None or res.aic < best[0]):
                    best = (res.aic, (p, d, q), tr, res)
    aic, order, tr, res = best
    lb = float(acorr_ljungbox(res.resid[order[1]:], lags=[10], return_df=True)["lb_pvalue"].iloc[0])
    return dict(order=order, trend=tr, aic=aic, adf_level_p=adf_lvl, adf_diff_p=adf_dif, ljungbox_p=lb)

def arima_rolling(y, order, trend, start, every=ARIMA_REFIT_EVERY):
    """One-step-ahead forecasts for targets start..n-1. Parameters re-estimated on all data before each block
    of `every` days; the state is updated with every new observation (no multi-step forecasting)."""
    n = len(y); out = np.empty(n - start); prev = None
    for b in range(start, n, every):
        e = min(b + every, n)
        mod = ARIMA(y[:b], order=order, trend=trend)
        try:
            res = mod.fit(start_params=prev) if prev is not None else mod.fit()
        except Exception:
            res = mod.fit()
        prev = res.params
        out[b - start:e - start] = res.apply(y[:e]).predict(start=b, end=e - 1)
    return out

BENCH, ARIMA_INFO = {}, {}
bench_file = RUN_DIR / "benchmarks.npz"
arima_file = RUN_DIR / "arima_info.json"
if bench_file.exists() and arima_file.exists():
    BENCH = {k: v for k, v in np.load(bench_file).items()}
    ARIMA_INFO = json.load(open(arima_file))
for t in TICKERS:
    if f"{t}|Persistence" in BENCH and t in ARIMA_INFO:
        continue
    y = SERIES[t].values; n = len(y); _, i_test = split_points(n); idx = np.arange(i_test, n)
    t0 = time.time()
    BENCH[f"{t}|Persistence"] = bench_persistence(y, idx)
    BENCH[f"{t}|Drift"] = bench_drift(y, idx)
    BENCH[f"{t}|Seasonal naive"] = bench_snaive(y, idx)
    BENCH[f"{t}|SES"], _ = bench_ets(y, i_test, idx, trend=False)
    BENCH[f"{t}|Holt"], _ = bench_ets(y, i_test, idx, trend=True)
    BENCH[f"{t}|Linear regression"] = bench_linreg(y, i_test, idx)
    info = arima_select(y[:i_test])
    BENCH[f"{t}|ARIMA"] = arima_rolling(y, info["order"], info["trend"], i_test)
    ARIMA_INFO[t] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in info.items()}
    np.savez(bench_file, **BENCH); json.dump(ARIMA_INFO, open(arima_file, "w"), indent=2)
    print(f"benchmarks {t}: ARIMA{tuple(info['order'])} trend={info['trend']}  ({time.time()-t0:.0f}s)")
BENCH_NAMES = ["Persistence", "Drift", "Seasonal naive", "SES", "Holt", "ARIMA", "Linear regression"]


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarnin

benchmarks TSLA: ARIMA(0, 1, 1) trend=n  (62s)


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarnin

benchmarks AMZN: ARIMA(5, 1, 2) trend=n  (165s)


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarnin

...

benchmarks IWM: ARIMA(4, 1, 2) trend=n  (116s)


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


benchmarks ^NSEI: ARIMA(4, 1, 2) trend=t  (156s)


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarnin

benchmarks RELIANCE.NS: ARIMA(3, 1, 2) trend=t  (166s)


In [ ]:
# ============================== 6. The two-layer network ==============================
from dataclasses import dataclass, asdict, replace

@dataclass(frozen=True)
class Cfg:
    name: str = "Base (L=30, 2x50, Dense 25)"
    L: int = 30
    units: int = 50
    layers: int = 2
    dense: int = 25
    batch: int = 8
    lr: float = 1e-3
    dropout: float = 0.0
    arch: str = "stacked"          # "stacked" or "flat" (LSTM -> Dense(50) ablation)

BASE_CFG = Cfg()

def build_model(c):
    inp = keras.Input(shape=(c.L, 1)); x = inp
    if c.arch == "flat":
        x = keras.layers.LSTM(c.units, return_sequences=False)(x)
        x = keras.layers.Dense(50)(x)
    else:
        for k in range(c.layers):
            x = keras.layers.LSTM(c.units, return_sequences=(k < c.layers - 1))(x)
            if c.dropout > 0 and k < c.layers - 1:
                x = keras.layers.Dropout(c.dropout)(x)
    x = keras.layers.Dense(c.dense)(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out)

def windows(ys, idx, L):
    return np.stack([ys[j - L:j] for j in idx])[..., None], ys[idx]

def run_key(exp, t, c, fold, seed):
    h = hashlib.md5(json.dumps(asdict(c), sort_keys=True).encode()).hexdigest()[:10]
    return f"{exp}__{_safe(t)}__{h}__f{fold}__s{seed}"

def train_eval(t, c, seed, train_end, val_end, pred_end, exp="main", fold=0):
    """Scaler on [0, train_end); fit on targets [L, train_end); early stopping on targets [train_end, val_end);
    returns de-scaled predictions for validation targets and for targets [val_end, pred_end)."""
    key = run_key(exp, t, c, fold, seed); f = RUN_DIR / f"{key}.npz"
    if f.exists():
        z = np.load(f, allow_pickle=True)
        return {k: z[k] for k in z.files} | {"meta": json.loads(str(z["meta"]))}
    y = SERIES[t].values
    mn, mx = y[:train_end].min(), y[:train_end].max()
    ys = (y - mn) / (mx - mn)
    tr, va, te = np.arange(c.L, train_end), np.arange(train_end, val_end), np.arange(val_end, pred_end)
    Xtr, Ttr = windows(ys, tr, c.L); Xva, Tva = windows(ys, va, c.L); Xte, _ = windows(ys, te, c.L)
    keras.backend.clear_session(); keras.utils.set_random_seed(seed)
    m = build_model(c)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=c.lr), loss="mse", steps_per_execution=32)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    t0 = time.time()
    h = m.fit(Xtr, Ttr, validation_data=(Xva, Tva), epochs=MAX_EPOCHS, batch_size=c.batch,
              shuffle=True, verbose=0, callbacks=[es])
    inv = lambda p: p.ravel() * (mx - mn) + mn
    res = dict(val_pred=inv(m.predict(Xva, batch_size=2048, verbose=0)),
               test_pred=inv(m.predict(Xte, batch_size=2048, verbose=0)),
               loss=np.array(h.history["loss"]), val_loss=np.array(h.history["val_loss"]))
    meta = dict(best_epoch=int(np.argmin(h.history["val_loss"]) + 1), epochs_run=len(h.history["loss"]),
                n_params=int(m.count_params()), seconds=round(time.time() - t0, 1), cfg=asdict(c), seed=seed)
    np.savez(f, **res, meta=json.dumps(meta))
    res["meta"] = meta
    return res


In [ ]:
# ============================== 7. Experiment A — static chronological split, all series, 5 seeds ==============================
MAIN = {}
total = len(TICKERS) * len(SEEDS_MAIN); k = 0; T0 = time.time()
for t in TICKERS:
    n = len(SERIES[t]); i_val, i_test = split_points(n)
    for s in SEEDS_MAIN:
        MAIN[(t, s)] = train_eval(t, BASE_CFG, s, i_val, i_test, n, exp="main")
        k += 1
        print(f"\r[A] {k}/{total}  {t} seed {s}  best epoch {MAIN[(t, s)]['meta']['best_epoch']}  "
              f"elapsed {(time.time()-T0)/60:.1f} min", end="")
print()

def test_arrays(t):
    y = SERIES[t].values; n = len(y); _, i_test = split_points(n); idx = np.arange(i_test, n)
    return y, idx, y[idx], y[idx - 1], naive_scale(y, i_test)

ROWS = []
for t in TICKERS:
    y, idx, yt, yp, sc = test_arrays(t)
    rp = float(np.sqrt(np.mean((yt - BENCH[f"{t}|Persistence"]) ** 2)))
    for b in BENCH_NAMES:
        ROWS.append(dict(ticker=t, model=b, seed=-1, **price_metrics(yt, BENCH[f"{t}|{b}"], yp, sc, rp)))
    for s in SEEDS_MAIN:
        ROWS.append(dict(ticker=t, model="LSTM", seed=s, **price_metrics(yt, MAIN[(t, s)]["test_pred"], yp, sc, rp)))
RES = pd.DataFrame(ROWS)
RES.to_csv(TAB_DIR / "all_test_metrics_long.csv", index=False)
LSTM_AGG = RES[RES.model == "LSTM"].drop(columns="model").groupby("ticker").agg(["mean", "std"])
LSTM_AGG = LSTM_AGG.reindex(list(TICKERS))


[A] 2/60  TSLA seed 1  best epoch 10  elapsed 3.1 min

[A] 12/60  GOOG seed 1  best epoch 14  elapsed 24.7 min

In [ ]:
# ============================== 8. Significance, direction, volatility regimes, trading ==============================
DM, VOL, TRADE = [], [], []
for t in TICKERS:
    y, idx, yt, yp, sc = test_arrays(t)
    n = len(y); _, i_test = split_points(n)
    r_all = np.concatenate([[np.nan], y[1:] / y[:-1] - 1])
    rv = pd.Series(r_all).rolling(VOL_WINDOW).std().shift(1).values          # uses returns up to t-1 only
    thr = np.nanpercentile(rv[VOL_WINDOW + 1:i_test], 75)                      # threshold fixed from pre-test data
    hi = rv[idx] > thr
    e_p = yt - BENCH[f"{t}|Persistence"]; e_a = yt - BENCH[f"{t}|ARIMA"]; e_h = yt - BENCH[f"{t}|Holt"]
    r = yt / yp - 1
    bh = trading_stats(r, np.ones_like(r), COST_BPS)
    for s in SEEDS_MAIN:
        e = yt - MAIN[(t, s)]["test_pred"]
        dp, pp = dm_test(e, e_p); da, pa = dm_test(e, e_a); dh, ph = dm_test(e, e_h)
        DM.append(dict(ticker=t, seed=s, dm_p=dp, p_p=pp, dm_a=da, p_a=pa, dm_h=dh, p_h=ph))
        rh = MAIN[(t, s)]["test_pred"] / yp - 1
        for regime, m in (("High volatility", hi), ("Normal", ~hi)):
            mm = m & (r != 0)
            VOL.append(dict(ticker=t, seed=s, regime=regime, share=float(m.mean() * 100),
                            nRMSE_lstm=float(np.sqrt(np.mean(e[m] ** 2)) / yt[m].mean() * 100),
                            nRMSE_pers=float(np.sqrt(np.mean(e_p[m] ** 2)) / yt[m].mean() * 100),
                            DA=float(np.mean(np.sign(rh[mm]) == np.sign(r[mm])) * 100)))
        pos = (rh > 0).astype(float)
        g = trading_stats(r, pos, 0); nt = trading_stats(r, pos, COST_BPS)
        TRADE.append(dict(ticker=t, seed=s, gross_ret=g["ann_ret"], net_ret=nt["ann_ret"], net_sharpe=nt["sharpe"],
                          net_mdd=nt["mdd"], trades=nt["trades"], bh_ret=bh["ann_ret"], bh_sharpe=bh["sharpe"], bh_mdd=bh["mdd"]))
DM, VOL, TRADE = pd.DataFrame(DM), pd.DataFrame(VOL), pd.DataFrame(TRADE)
for nm, d in (("dm_tests", DM), ("volatility_regimes", VOL), ("trading", TRADE)):
    d.to_csv(TAB_DIR / f"{nm}_by_seed.csv", index=False)


In [ ]:
# ============================== 9. Experiment B — expanding-window walk-forward ==============================
WF = []
wf_total = len(TICKERS) * N_WF_FOLDS * len(SEEDS_WF); k = 0
FOLD_DATES = {}
for t in TICKERS:
    y = SERIES[t].values; n = len(y); _, i_test = split_points(n)
    b = np.linspace(i_test, n, N_WF_FOLDS + 1).astype(int)
    FOLD_DATES[t] = [(SERIES[t].index[b[f]].date(), SERIES[t].index[b[f + 1] - 1].date()) for f in range(N_WF_FOLDS)]
    for f in range(N_WF_FOLDS):
        s0, s1 = b[f], b[f + 1]
        tr_end = split_points(n)[0] if f == 0 else int(s0 * TRAIN_FRAC / (TRAIN_FRAC + VAL_FRAC))   # same 7:1 train/validation ratio as Experiment A
        idx = np.arange(s0, s1); yt, yp = y[idx], y[idx - 1]; sc = naive_scale(y, s0)
        preds = {"Persistence": y[idx - 1], "ARIMA": BENCH[f"{t}|ARIMA"][s0 - i_test:s1 - i_test]}
        preds["Holt"], _ = bench_ets(y, s0, idx, trend=True)
        preds["Linear regression"] = bench_linreg(y, s0, idx)
        rp = float(np.sqrt(np.mean((yt - preds["Persistence"]) ** 2)))
        for nm, p in preds.items():
            WF.append(dict(ticker=t, fold=f + 1, model=nm, seed=-1, **price_metrics(yt, p, yp, sc, rp)))
        for s in SEEDS_WF:
            if f == 0 and s in SEEDS_MAIN:          # fold 1 has exactly the Experiment-A training data
                p = MAIN[(t, s)]["test_pred"][:s1 - s0]
            else:
                p = train_eval(t, BASE_CFG, s, tr_end, s0, s1, exp="wf", fold=f + 1)["test_pred"]
            e = yt - p
            dstat, dp = dm_test(e, yt - preds["Persistence"])
            WF.append(dict(ticker=t, fold=f + 1, model="LSTM", seed=s, dm=dstat, dm_p=dp,
                           **price_metrics(yt, p, yp, sc, rp)))
            k += 1
            print(f"\r[B] {k}/{wf_total}  {t} fold {f+1} seed {s}", end="")
print()
WF = pd.DataFrame(WF); WF.to_csv(TAB_DIR / "walk_forward_long.csv", index=False)


In [ ]:
# ============================== 10. Experiment C — sensitivity (validation only) and ablation ==============================
SENS_CFGS = [BASE_CFG]
SENS_CFGS += [replace(BASE_CFG, name=f"Look-back {L}", L=L) for L in (10, 20, 45, 60)]
SENS_CFGS += [replace(BASE_CFG, name=f"Units {u}", units=u) for u in (25, 100)]
SENS_CFGS += [replace(BASE_CFG, name=f"Dense {d}", dense=d) for d in (10, 50)]
SENS_CFGS += [replace(BASE_CFG, name=f"Batch {bs}", batch=bs) for bs in (16, 32)]
SENS_CFGS += [replace(BASE_CFG, name=f"Learning rate {lr:g}", lr=lr) for lr in (1e-4, 1e-2)]
SENS_CFGS += [replace(BASE_CFG, name=f"Dropout {dr}", dropout=dr) for dr in (0.1, 0.2, 0.3)]
ABL_CFGS = [replace(BASE_CFG, name="One layer, 50 units", layers=1),
            replace(BASE_CFG, name="One layer, 100 units", layers=1, units=100),
            replace(BASE_CFG, name="One layer, 85 units (parameter-matched)", layers=1, units=85),
            replace(BASE_CFG, name="LSTM(50) + Dense(50), no second recurrent layer", arch="flat")]
SENS = []
all_cfgs = SENS_CFGS + ABL_CFGS
c_total = len(all_cfgs) * len(CORE) * len(SEEDS_SENS); k = 0
for c in all_cfgs:
    for t in CORE:
        y = SERIES[t].values; n = len(y); i_val, i_test = split_points(n)
        va_idx = np.arange(i_val, i_test); te_idx = np.arange(i_test, n)
        for s in SEEDS_SENS:
            r = MAIN[(t, s)] if (c == BASE_CFG and (t, s) in MAIN) else train_eval(t, c, s, i_val, i_test, n, exp="sens")
            yv, yt = y[va_idx], y[te_idx]
            SENS.append(dict(config=c.name, ticker=t, seed=s, params=r["meta"]["n_params"],
                             best_epoch=r["meta"]["best_epoch"],
                             val_nRMSE=float(np.sqrt(np.mean((yv - r["val_pred"]) ** 2)) / yv.mean() * 100),
                             test_nRMSE=float(np.sqrt(np.mean((yt - r["test_pred"]) ** 2)) / yt.mean() * 100),
                             test_MASE=float(np.mean(np.abs(yt - r["test_pred"])) / naive_scale(y, i_test)),
                             val_nRMSE_pers=float(np.sqrt(np.mean((yv - y[va_idx - 1]) ** 2)) / yv.mean() * 100)))
            k += 1
            print(f"\r[C] {k}/{c_total}  {c.name} {t}", end="")
print()
SENS = pd.DataFrame(SENS); SENS.to_csv(TAB_DIR / "sensitivity_ablation_long.csv", index=False)


In [ ]:
# ============================== 11. Figures ==============================
FIGS = {}
def savefig(fig, name):
    p300, p600 = FIG_DIR / f"{name}.png", FIG_DIR / f"{name}_600dpi.png"
    fig.savefig(p300, dpi=300, bbox_inches="tight"); fig.savefig(p600, dpi=600, bbox_inches="tight")
    plt.close(fig); FIGS[name] = p300

def lstm_mean_pred(t): return np.mean([MAIN[(t, s)]["test_pred"] for s in SEEDS_MAIN], axis=0)
def label(t): return f"{t} ({TICKERS[t][0]})"
TL = list(TICKERS); ncol = 3; nrow = int(np.ceil(len(TL) / ncol))

# Fig: split scheme (TSLA if available)
t = "TSLA" if "TSLA" in TICKERS else TL[0]
s = SERIES[t]; n = len(s); i_val, i_test = split_points(n); b = np.linspace(i_test, n, N_WF_FOLDS + 1).astype(int)
fig, ax = plt.subplots(2, 1, figsize=(7.0, 3.6), gridspec_kw={"height_ratios": [3, 1.4]}, sharex=True)
ax[0].plot(s.index, s.values, color=C_ACT, lw=0.7)
ax[0].axvspan(s.index[0], s.index[i_val - 1], color="#9ecae1", alpha=0.35, label="Training (70 %)")
ax[0].axvspan(s.index[i_val], s.index[i_test - 1], color="#fdd49e", alpha=0.6, label="Validation (10 %)")
ax[0].axvspan(s.index[i_test], s.index[-1], color="#c7e9c0", alpha=0.6, label="Test (20 %)")
ax[0].set_ylabel(f"{t} opening price ({TICKERS[t][3]})"); ax[0].legend(loc="upper left", ncol=3, frameon=False)
for f in range(N_WF_FOLDS):
    s0, s1 = b[f], b[f + 1]; trn = int(s0 * TRAIN_FRAC / (TRAIN_FRAC + VAL_FRAC)); yy = N_WF_FOLDS - f
    ax[1].barh(yy, (s.index[trn - 1] - s.index[0]).days, left=s.index[0], color="#9ecae1", height=0.6)
    ax[1].barh(yy, (s.index[s0 - 1] - s.index[trn]).days, left=s.index[trn], color="#fdd49e", height=0.6)
    ax[1].barh(yy, (s.index[s1 - 1] - s.index[s0]).days, left=s.index[s0], color="#31a354", height=0.6)
ax[1].set_yticks(range(1, N_WF_FOLDS + 1)); ax[1].set_yticklabels([f"Fold {N_WF_FOLDS - i + 1}" for i in range(1, N_WF_FOLDS + 1)])
ax[1].set_xlabel("Date"); ax[1].grid(False)
fig.tight_layout(); savefig(fig, "fig_splits")

# Fig: actual vs predicted, all series
fig, axs = plt.subplots(nrow, ncol, figsize=(7.2, 1.9 * nrow))
for a, t in zip(axs.ravel(), TL):
    y, idx, yt, yp, sc = test_arrays(t); d = SERIES[t].index[idx]
    a.plot(d, yt, color=C_ACT, lw=0.8, label="Actual"); a.plot(d, lstm_mean_pred(t), color=C_LSTM, lw=0.8, alpha=0.9, label="Network (seed mean)")
    a.set_title(label(t)); a.set_ylabel(TICKERS[t][3]); a.xaxis.set_major_locator(mdates.YearLocator()); a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
for a in axs.ravel()[len(TL):]: a.axis("off")
axs.ravel()[0].legend(frameon=False, loc="upper left")
fig.tight_layout(); savefig(fig, "fig_actual_vs_pred")

# Fig: last 60 test days, network vs persistence (shows lag)
zt = [t for t in (CORE + ["^GSPC"]) if t in TICKERS][:6]
fig, axs = plt.subplots(2, 3, figsize=(7.2, 4.0))
for a, t in zip(axs.ravel(), zt):
    y, idx, yt, yp, sc = test_arrays(t); d = SERIES[t].index[idx][-60:]
    a.plot(d, yt[-60:], "-o", ms=1.6, color=C_ACT, lw=0.8, label="Actual")
    a.plot(d, lstm_mean_pred(t)[-60:], "-", color=C_LSTM, lw=0.9, label="Network")
    a.plot(d, BENCH[f"{t}|Persistence"][-60:], "--", color=C_PER, lw=0.9, label="Persistence")
    a.set_title(t); a.set_ylabel(TICKERS[t][3]); a.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
for a in axs.ravel()[len(zt):]: a.axis("off")
axs.ravel()[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); savefig(fig, "fig_zoom")

# Fig: directional accuracy with 95 % CI
fig, ax = plt.subplots(figsize=(7.0, 2.8))
xs = np.arange(len(TL)); vals, lo, hi_ = [], [], []
for t in TL:
    y, idx, yt, yp, sc = test_arrays(t); r = yt / yp - 1; rh = lstm_mean_pred(t) / yp - 1; m = (r != 0)
    kk = int(np.sum(np.sign(r[m]) == np.sign(rh[m]))); nn = int(m.sum()); l, u = wilson_ci(kk, nn)
    vals.append(kk / nn * 100); lo.append(kk / nn * 100 - l * 100); hi_.append(u * 100 - kk / nn * 100)
ax.bar(xs, vals, color=C_LSTM, alpha=0.8, yerr=[lo, hi_], capsize=3, ecolor="#333333")
ax.axhline(50, color="#aa0000", ls="--", lw=0.9, label="50 % (coin flip)")
ax.set_xticks(xs); ax.set_xticklabels(TL, rotation=30, ha="right"); ax.set_ylabel("Directional accuracy (%)")
ax.set_ylim(min(35, min(vals) - 8), max(65, max(vals) + 8)); ax.legend(frameon=False)
fig.tight_layout(); savefig(fig, "fig_direction")

# Fig: error distributions
fig, axs = plt.subplots(nrow, ncol, figsize=(7.2, 1.8 * nrow))
for a, t in zip(axs.ravel(), TL):
    y, idx, yt, yp, sc = test_arrays(t)
    el = (yt - lstm_mean_pred(t)) / yt * 100; ep = (yt - BENCH[f"{t}|Persistence"]) / yt * 100
    lim = np.nanpercentile(np.abs(np.concatenate([el, ep])), 99); bins = np.linspace(-lim, lim, 50)
    a.hist(ep, bins=bins, color=C_PER, alpha=0.5, density=True, label="Persistence")
    a.hist(el, bins=bins, color=C_LSTM, alpha=0.5, density=True, label="Network")
    a.axvline(np.mean(el), color=C_LSTM, lw=0.9); a.set_title(t); a.set_xlabel("Error (% of price)")
for a in axs.ravel()[len(TL):]: a.axis("off")
axs.ravel()[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); savefig(fig, "fig_errors")

# Fig: walk-forward
wfm = WF.groupby(["ticker", "fold", "model"])["MASE"].mean().reset_index()
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={"width_ratios": [1.1, 1]})
mods = ["LSTM", "Persistence", "ARIMA", "Holt"]; cols = [C_LSTM, C_PER, C_ARIMA, C_HOLT]; w = 0.2
for i, (mname, col) in enumerate(zip(mods, cols)):
    v = wfm[wfm.model == mname].groupby("fold")["MASE"].agg(["mean", "std"])
    axs[0].bar(v.index + (i - 1.5) * w, v["mean"], w, yerr=v["std"], capsize=2, color=col, label=mname if mname != "LSTM" else "Network")
axs[0].set_xlabel("Walk-forward fold"); axs[0].set_ylabel("MASE (mean over series)"); axs[0].legend(frameon=False, fontsize=7)
axs[0].set_xticks(range(1, N_WF_FOLDS + 1))
u2 = WF[WF.model == "LSTM"].groupby(["ticker", "fold"])["U2"].mean().unstack().reindex(TL)
im = axs[1].imshow(u2.values, cmap="RdBu_r", vmin=0.8, vmax=1.2, aspect="auto")
axs[1].set_yticks(range(len(TL))); axs[1].set_yticklabels(TL); axs[1].set_xticks(range(N_WF_FOLDS)); axs[1].set_xticklabels(range(1, N_WF_FOLDS + 1))
axs[1].set_xlabel("Fold"); axs[1].grid(False)
for i in range(u2.shape[0]):
    for j in range(u2.shape[1]):
        axs[1].text(j, i, f"{u2.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=axs[1], label="Theil U2 (network / persistence RMSE)")
fig.tight_layout(); savefig(fig, "fig_walkforward")

# Fig: sensitivity (validation)
sv = SENS.groupby("config")["val_nRMSE"].agg(["mean", "std"])
groups = [("Look-back (days)", [(10, "Look-back 10"), (20, "Look-back 20"), (30, BASE_CFG.name), (45, "Look-back 45"), (60, "Look-back 60")]),
          ("Units per layer", [(25, "Units 25"), (50, BASE_CFG.name), (100, "Units 100")]),
          ("Dense units", [(10, "Dense 10"), (25, BASE_CFG.name), (50, "Dense 50")]),
          ("Batch size", [(8, BASE_CFG.name), (16, "Batch 16"), (32, "Batch 32")]),
          ("Learning rate", [(1e-4, "Learning rate 0.0001"), (1e-3, BASE_CFG.name), (1e-2, "Learning rate 0.01")]),
          ("Dropout", [(0.0, BASE_CFG.name), (0.1, "Dropout 0.1"), (0.2, "Dropout 0.2"), (0.3, "Dropout 0.3")])]
pers_val = SENS[SENS.config == BASE_CFG.name]["val_nRMSE_pers"].mean()
fig, axs = plt.subplots(2, 3, figsize=(7.2, 4.0))
for a, (gname, items) in zip(axs.ravel(), groups):
    xv = [x for x, _ in items]; yv = [sv.loc[c, "mean"] for _, c in items]; ev = [sv.loc[c, "std"] for _, c in items]
    a.errorbar(range(len(xv)), yv, yerr=ev, marker="o", ms=3, color=C_LSTM, capsize=2)
    a.axhline(pers_val, color=C_PER, ls="--", lw=0.8, label="Persistence")
    a.set_xticks(range(len(xv))); a.set_xticklabels([f"{x:g}" for x in xv]); a.set_title(gname); a.set_ylabel("Validation nRMSE (%)")
axs.ravel()[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); savefig(fig, "fig_sensitivity")

# Fig: learning curves
fig, axs = plt.subplots(nrow, ncol, figsize=(7.2, 1.8 * nrow))
for a, t in zip(axs.ravel(), TL):
    r = MAIN[(t, SEEDS_MAIN[0])]
    a.semilogy(np.arange(1, len(r["loss"]) + 1), r["loss"], color=C_LSTM, lw=0.9, label="Training")
    a.semilogy(np.arange(1, len(r["val_loss"]) + 1), r["val_loss"], color=C_PER, lw=0.9, label="Validation")
    a.axvline(r["meta"]["best_epoch"], color="#555555", ls=":", lw=0.8); a.set_title(t); a.set_xlabel("Epoch")
for a in axs.ravel()[len(TL):]: a.axis("off")
axs.ravel()[0].set_ylabel("MSE (scaled)"); axs.ravel()[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); savefig(fig, "fig_learning_curves")

# Fig: equity curves
fig, axs = plt.subplots(nrow, ncol, figsize=(7.2, 1.8 * nrow))
for a, t in zip(axs.ravel(), TL):
    y, idx, yt, yp, sc = test_arrays(t); r = yt / yp - 1; d = SERIES[t].index[idx]
    a.plot(d, trading_stats(r, np.ones_like(r), COST_BPS)["equity"], color=C_ACT, lw=0.9, label="Buy and hold")
    for i, s in enumerate(SEEDS_MAIN):
        pos = (MAIN[(t, s)]["test_pred"] / yp - 1 > 0).astype(float)
        a.plot(d, trading_stats(r, pos, COST_BPS)["equity"], color=C_LSTM, lw=0.6, alpha=0.6, label="Network long/flat" if i == 0 else None)
    a.set_title(t); a.xaxis.set_major_locator(mdates.YearLocator()); a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
for a in axs.ravel()[len(TL):]: a.axis("off")
axs.ravel()[0].set_ylabel("Growth of 1 unit"); axs.ravel()[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); savefig(fig, "fig_equity")
print("Figures:", ", ".join(FIGS))


In [ ]:
# ============================== 12. Tables ==============================
def fm(x, d=2):
    return "–" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"
def ms(m, s, d=2): return f"{m:.{d}f} ± {s:.{d}f}"
def pfmt(p):
    if p is None or not np.isfinite(p): return "n/a"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

TABLES = {}
TABLES["data"] = (["Symbol", "Name", "Sector / type", "Country", "Unit", "Trading days", "Removed", "Training", "Validation", "Test"],
                  [[r.ticker, r["name"], r.sector, r.country, r.unit, str(r.n_used), str(r.n_missing),
                    f"{r.train} ({r.n_train})", f"{r.val} ({r.n_val})", f"{r.test} ({r.n_test})"] for _, r in QUALITY.iterrows()])
tsla = RAW["TSLA"] if "TSLA" in RAW else RAW[TL[0]]
smp = pd.concat([tsla.head(3), tsla.tail(3)])
TABLES["sample"] = (["Date", "Open", "High", "Low", "Close", "Volume"],
                    [[str(i.date()), f"{r.Open:.4f}", f"{r.High:.4f}", f"{r.Low:.4f}", f"{r.Close:.4f}", f"{int(r.Volume):,}"] for i, r in smp.iterrows()])

import importlib
def _ver(m):
    try: return importlib.import_module(m).__version__
    except Exception: return "n/a"
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "none (CPU)"
except Exception:
    gpu = "none (CPU)"
try:
    import psutil; ram = f"{psutil.virtual_memory().total / 1e9:.1f} GB"
except Exception:
    ram = "n/a"
TABLES["env"] = (["Item", "Value"], [
    ["Platform", "Google Colab" if IN_COLAB else platform.platform()], ["Python", platform.python_version()],
    ["TensorFlow / Keras", f"{tf.__version__} / {keras.__version__}"], ["NumPy / pandas", f"{np.__version__} / {pd.__version__}"],
    ["statsmodels / scikit-learn", f"{statsmodels.__version__} / {_ver('sklearn')}"], ["yfinance", DL_INFO["yfinance_version"]],
    ["GPU", gpu], ["CPU", platform.processor() or "virtualised x86-64"], ["System memory", ram],
    ["Data download (UTC)", DL_INFO["download_date_utc"]], ["Download call", DL_INFO["settings"]],
    ["Random seeds", ", ".join(map(str, SEEDS_MAIN)) + " (walk-forward: " + ", ".join(map(str, SEEDS_WF)) + ")"],
    ["Early stopping", f"monitor = validation loss, patience = {PATIENCE}, restore best weights = true, max epochs = {MAX_EPOCHS}"]])

TABLES["arima"] = (["Symbol", "ADF p (level)", "ADF p (1st diff.)", "Selected order", "Trend term", "AIC", "Ljung–Box p (lag 10)", "SES α", "Holt α / β"], [])
for t in TL:
    a = ARIMA_INFO[t]; y = SERIES[t].values; _, i_test = split_points(len(y))
    _, pa = bench_ets(y, i_test, np.array([len(y) - 1]), False); _, ph = bench_ets(y, i_test, np.array([len(y) - 1]), True)
    TABLES["arima"][1].append([t, pfmt(a["adf_level_p"]), pfmt(a["adf_diff_p"]), f"({a['order'][0]},{a['order'][1]},{a['order'][2]})",
                               {"n": "none", "t": "drift", "c": "constant"}[a["trend"]], f"{a['aic']:.0f}", pfmt(a["ljungbox_p"]),
                               f"{pa['alpha']:.3f}", f"{ph['alpha']:.3f} / {ph['beta']:.3f}"])

TABLES["lstm_main"] = (["Symbol", "RMSE", "MAE", "MAPE (%)", "nRMSE (%)", "MASE", "R²", "Best epoch"], [])
for t in TL:
    g = LSTM_AGG.loc[t]; be = np.mean([MAIN[(t, s)]["meta"]["best_epoch"] for s in SEEDS_MAIN])
    TABLES["lstm_main"][1].append([t, ms(g[("RMSE", "mean")], g[("RMSE", "std")]), ms(g[("MAE", "mean")], g[("MAE", "std")]),
                                   ms(g[("MAPE", "mean")], g[("MAPE", "std")]), ms(g[("nRMSE", "mean")], g[("nRMSE", "std")]),
                                   ms(g[("MASE", "mean")], g[("MASE", "std")], 3), ms(g[("R2", "mean")], g[("R2", "std")], 4), f"{be:.0f}"])

bm = RES[RES.model != "LSTM"].set_index(["ticker", "model"])
TABLES["bench"] = (["Symbol"] + BENCH_NAMES + ["Network (mean ± sd)", "Network U2"], [])
for t in TL:
    vals = [bm.loc[(t, b), "MASE"] for b in BENCH_NAMES]
    best = min(vals + [LSTM_AGG.loc[t, ("MASE", "mean")]])
    row = [t] + [fm(v, 3) + (" *" if abs(v - best) < 1e-12 else "") for v in vals]
    lm = LSTM_AGG.loc[t, ("MASE", "mean")]
    row += [ms(lm, LSTM_AGG.loc[t, ("MASE", "std")], 3) + (" *" if abs(lm - best) < 1e-12 else ""), fm(LSTM_AGG.loc[t, ("U2", "mean")], 3)]
    TABLES["bench"][1].append(row)

dmg = DM.groupby("ticker").median().reindex(TL)
dm_sig = DM.assign(sb=(DM.dm_p < 0) & (DM.p_p < 0.05), sw=(DM.dm_p > 0) & (DM.p_p < 0.05)).groupby("ticker")[["sb", "sw"]].sum().reindex(TL)
TABLES["dm"] = (["Symbol", "vs persistence: DM", "p", "Seeds sig. better / worse", "vs ARIMA: DM", "p", "vs Holt: DM", "p"], [])
for t in TL:
    r = dmg.loc[t]
    TABLES["dm"][1].append([t, fm(r.dm_p), pfmt(r.p_p), f"{int(dm_sig.loc[t, 'sb'])} / {int(dm_sig.loc[t, 'sw'])} of {len(SEEDS_MAIN)}",
                            fm(r.dm_a), pfmt(r.p_a), fm(r.dm_h), pfmt(r.p_h)])

TABLES["returns"] = (["Symbol", "Return RMSE, network (%)", "Return RMSE, persistence (%)", "R²oos on returns (%)",
                      "Directional accuracy (%)", "Up calls (%)", "PT p (median)"], [])
lm_all = RES[RES.model == "LSTM"]
for t in TL:
    g = lm_all[lm_all.ticker == t]; per = bm.loc[(t, "Persistence")]
    TABLES["returns"][1].append([t, ms(g.retRMSE.mean(), g.retRMSE.std(), 3), fm(per.retRMSE, 3), ms(g.R2oos.mean(), g.R2oos.std()),
                                 ms(g.DA.mean(), g.DA.std(), 1), fm(g.up_share.mean(), 1), pfmt(g.PT_p.median())])

vg = VOL.groupby(["ticker", "regime"]).mean(numeric_only=True)
TABLES["vol"] = (["Symbol", "High-vol. days (%)", "nRMSE high: network", "nRMSE high: persistence", "DA high (%)",
                  "nRMSE normal: network", "nRMSE normal: persistence", "DA normal (%)"], [])
for t in TL:
    h, nm = vg.loc[(t, "High volatility")], vg.loc[(t, "Normal")]
    TABLES["vol"][1].append([t, fm(h.share, 1), fm(h.nRMSE_lstm), fm(h.nRMSE_pers), fm(h.DA, 1), fm(nm.nRMSE_lstm), fm(nm.nRMSE_pers), fm(nm.DA, 1)])

TABLES["wf_folds"] = (["Fold", "Test window (TSLA calendar)", "MASE network", "MASE persistence", "MASE ARIMA", "MASE Holt",
                       "Series with U2 < 1", "Series with DM p < 0.05 (better)", "DA network (%)"], [])
ref = "TSLA" if "TSLA" in TICKERS else TL[0]
for fo in range(1, N_WF_FOLDS + 1):
    w = WF[WF.fold == fo]; wl = w[w.model == "LSTM"].groupby("ticker").mean(numeric_only=True)
    sig = w[w.model == "LSTM"].groupby("ticker").apply(lambda g: np.median(g.dm) < 0 and np.median(g.dm_p) < 0.05).sum()
    TABLES["wf_folds"][1].append([str(fo), f"{FOLD_DATES[ref][fo-1][0]} – {FOLD_DATES[ref][fo-1][1]}",
                                  fm(wl.MASE.mean(), 3), fm(w[w.model == 'Persistence'].MASE.mean(), 3), fm(w[w.model == 'ARIMA'].MASE.mean(), 3),
                                  fm(w[w.model == 'Holt'].MASE.mean(), 3), f"{int((wl.U2 < 1).sum())} / {len(wl)}", f"{int(sig)} / {len(wl)}", fm(wl.DA.mean(), 1)])

sg = SENS.groupby("config").agg(params=("params", "first"), val=("val_nRMSE", "mean"), val_sd=("val_nRMSE", "std"),
                                ep=("best_epoch", "mean"), test=("test_nRMSE", "mean"), mase=("test_MASE", "mean"))
TABLES["sens"] = (["Configuration (one factor changed)", "Parameters", "Validation nRMSE (%) mean ± sd over series", "Mean best epoch"], [])
for c in SENS_CFGS:
    r = sg.loc[c.name]
    TABLES["sens"][1].append([c.name + (" (reference)" if c == BASE_CFG else ""), f"{int(r.params):,}", ms(r.val, r.val_sd, 3), fm(r.ep, 0)])
TABLES["sens"][1].append(["Persistence (for reference)", "0", fm(pers_val, 3), "–"])

TABLES["ablation"] = (["Variant", "Parameters", "Validation nRMSE (%)", "Test nRMSE (%)", "Test MASE"] + [f"Test MASE {t}" for t in CORE], [])
per_t = SENS.groupby(["config", "ticker"])["test_MASE"].mean()
for c in [BASE_CFG] + ABL_CFGS:
    r = sg.loc[c.name]
    TABLES["ablation"][1].append([c.name, f"{int(r.params):,}", fm(r.val, 3), fm(r.test, 3), fm(r.mase, 3)] + [fm(per_t.loc[(c.name, t)], 3) for t in CORE])

tg = TRADE.groupby("ticker").agg(["mean", "std"]).reindex(TL)
TABLES["trading"] = (["Symbol", "Buy & hold: ann. return (%)", "Sharpe", "Max DD (%)", "Network gross ann. return (%)",
                      f"Network net ({COST_BPS} bps) ann. return (%)", "Net Sharpe", "Net max DD (%)", "Position changes"], [])
for t in TL:
    g = tg.loc[t]
    TABLES["trading"][1].append([t, fm(g[("bh_ret", "mean")], 1), fm(g[("bh_sharpe", "mean")]), fm(g[("bh_mdd", "mean")], 1),
                                 ms(g[("gross_ret", "mean")], g[("gross_ret", "std")], 1), ms(g[("net_ret", "mean")], g[("net_ret", "std")], 1),
                                 ms(g[("net_sharpe", "mean")], g[("net_sharpe", "std")]), fm(g[("net_mdd", "mean")], 1), fm(g[("trades", "mean")], 0)])

for k_, (h, rows) in TABLES.items():
    pd.DataFrame(rows, columns=h).to_csv(TAB_DIR / f"table_{k_}.csv", index=False)
print("Tables:", ", ".join(TABLES))


In [ ]:
# ============================== 13. Numbers and sentences for the text ==============================
N = len(TL)
lstm_u2 = LSTM_AGG[("U2", "mean")]
n_u2_better = int((lstm_u2 < 1).sum())
n_sig_better = int(((dmg.dm_p < 0) & (dmg.p_p < 0.05)).sum())
n_sig_worse = int(((dmg.dm_p > 0) & (dmg.p_p < 0.05)).sum())
n_sig_better_arima = int(((dmg.dm_a < 0) & (dmg.p_a < 0.05)).sum())
n_sig_worse_arima = int(((dmg.dm_a > 0) & (dmg.p_a < 0.05)).sum())
da_mean = float(lm_all.groupby("ticker").DA.mean().mean())
da_min, da_max = float(lm_all.groupby("ticker").DA.mean().min()), float(lm_all.groupby("ticker").DA.mean().max())
n_pt_sig = int((lm_all.groupby("ticker").PT_p.median() < 0.05).sum())
r2oos_mean = float(lm_all.groupby("ticker").R2oos.mean().mean())
up_mean = float(lm_all.groupby("ticker").up_share.mean().mean())
r2_min = float(LSTM_AGG[("R2", "mean")].min())
mape_min, mape_max = float(LSTM_AGG[("MAPE", "mean")].min()), float(LSTM_AGG[("MAPE", "mean")].max())
mase_min, mase_max = float(LSTM_AGG[("MASE", "mean")].min()), float(LSTM_AGG[("MASE", "mean")].max())
pers_mase = bm.xs("Persistence", level="model")["MASE"]
u2_min, u2_max = float(lstm_u2.min()), float(lstm_u2.max())
best_names = []
for t in TL:
    cand = {b: bm.loc[(t, b), "MASE"] for b in BENCH_NAMES}; cand["Network"] = LSTM_AGG.loc[t, ("MASE", "mean")]
    best_names.append(min(cand, key=cand.get))
best_model_count = pd.Series(best_names).value_counts()
best_model_text = ", ".join(f"{m} ({c})" for m, c in best_model_count.items())
wfl = WF[WF.model == "LSTM"].groupby(["ticker", "fold"]).mean(numeric_only=True)
wf_cells, wf_better = len(wfl), int((wfl.U2 < 1).sum())
vg_h = vg.xs("High volatility", level="regime"); vg_n = vg.xs("Normal", level="regime")
n_hi_better = int((vg_h.nRMSE_lstm < vg_h.nRMSE_pers).sum())
hi_ratio = float((vg_h.nRMSE_lstm / vg_h.nRMSE_pers).mean()); no_ratio = float((vg_n.nRMSE_lstm / vg_n.nRMSE_pers).mean())
n_trade_beat = int((tg[("net_sharpe", "mean")] > tg[("bh_sharpe", "mean")]).sum())
gross_minus_net = float((tg[("gross_ret", "mean")] - tg[("net_ret", "mean")]).mean())
sv_best = sg.loc[[c.name for c in SENS_CFGS]].val.idxmin()
base_val = float(sg.loc[BASE_CFG.name].val)
best_val = float(sg.loc[sv_best].val)
lb_best = sg.loc[[c.name for c in SENS_CFGS if c.name.startswith("Look-back") or c == BASE_CFG]].val.idxmin()
lb_best_txt = "30 days" if lb_best == BASE_CFG.name else lb_best.replace("Look-back ", "") + " days"
abl = sg.loc[[c.name for c in [BASE_CFG] + ABL_CFGS]]
matched = sg.loc["One layer, 85 units (parameter-matched)"]
gap_ratio = float(np.mean([MAIN[(t, s)]["val_loss"].min() / MAIN[(t, s)]["loss"][MAIN[(t, s)]["meta"]["best_epoch"] - 1]
                           for t in TL for s in SEEDS_MAIN]))
seed_cv = float((LSTM_AGG[("RMSE", "std")] / LSTM_AGG[("RMSE", "mean")]).mean() * 100)
lb_fail = int(sum(ARIMA_INFO[t]["ljungbox_p"] < 0.05 for t in TL))
orders = ", ".join(f"{t} ({ARIMA_INFO[t]['order'][0]},{ARIMA_INFO[t]['order'][1]},{ARIMA_INFO[t]['order'][2]})" for t in TL)
n_arima_vs_p_better = int(sum(bm.loc[(t, "ARIMA"), "RMSE"] < bm.loc[(t, "Persistence"), "RMSE"] for t in TL))
core_u2 = float(lstm_u2.reindex(CORE).mean()); ext = [t for t in TL if t not in CORE]
ext_u2 = float(lstm_u2.reindex(ext).mean()) if ext else float("nan")

def words(k):
    w = "zero one two three four five six seven eight nine ten eleven twelve".split()
    return w[k] if 0 <= k < len(w) else str(k)

if n_sig_better >= math.ceil(N / 2):
    s_dm = (f"The network was significantly more accurate than the random-walk forecast for {words(n_sig_better)} of the "
            f"{words(N)} series (Diebold–Mariano test, 5 % level)")
elif n_sig_better > 0:
    s_dm = (f"The network was significantly more accurate than the random-walk forecast for only {words(n_sig_better)} of the "
            f"{words(N)} series (Diebold–Mariano test, 5 % level)")
else:
    s_dm = (f"The network was not significantly more accurate than the random-walk forecast for any of the {words(N)} series "
            f"(Diebold–Mariano test, 5 % level)")
s_dm += (f", and it was significantly less accurate for {words(n_sig_worse)}." if n_sig_worse > 0 else ".")

gain = n_sig_better >= math.ceil(N / 2) and r2oos_mean > 0
if gain:
    s_infer = ("The network therefore extracts a small but measurable amount of short-term structure beyond the last observed price, "
               "although the size of the gain is modest compared with what price-level scores suggest.")
    s_infer_plain = ("a carefully validated two-layer recurrent network adds a small but statistically detectable improvement over simply "
                     "repeating the last price, but price-level scores greatly overstate how much it adds")
else:
    s_infer = ("Very high coefficients of determination on price levels therefore mostly reflect the persistence of prices: at a one-day "
               "horizon a univariate recurrent network adds little or no information beyond the last observed price.")
    s_infer_plain = ("very high price-level scores mainly reflect the strong day-to-day persistence of prices, and a univariate two-layer "
                     "recurrent network adds little or no reliable information beyond the last observed price at a one-day horizon")

if n_pt_sig == 0:
    s_da = (f"Directional accuracy averaged {da_mean:.1f} % (range {da_min:.1f}–{da_max:.1f} %), and the Pesaran–Timmermann test did not "
            f"reject the no-skill hypothesis for any series.")
else:
    s_da = (f"Directional accuracy averaged {da_mean:.1f} % (range {da_min:.1f}–{da_max:.1f} %); the Pesaran–Timmermann test rejected the "
            f"no-skill hypothesis for {words(n_pt_sig)} of the {words(N)} series.")
s_up = (f" On average {up_mean:.0f} % of the network's calls were 'up', so part of its hit rate comes from the upward drift of the test period "
        f"rather than from timing." if up_mean > 60 else "")
s_trade = (f"After a one-way cost of {COST_BPS} basis points, the long/flat rule built on the network's forecasts had a higher Sharpe ratio "
           f"than buy-and-hold for {words(n_trade_beat)} of the {words(N)} series; trading costs reduced the annual return by "
           f"{gross_minus_net:.1f} percentage points on average.")
s_wf = (f"Across the {wf_cells} series-fold combinations of the walk-forward test, the network's RMSE was below that of persistence in "
        f"{wf_better} cases ({wf_better / wf_cells * 100:.0f} %).")
s_vol = (f"On high-volatility days the network's nRMSE averaged {hi_ratio:.2f} times that of persistence, against {no_ratio:.2f} times on "
         f"normal days; it beat persistence on high-volatility days for {words(n_hi_better)} of the {words(N)} series.")
s_sens = ("The reference configuration also gave the lowest mean validation error, so the original choice is confirmed without "
          "looking at the test set." if sv_best == BASE_CFG.name else
          f"The lowest mean validation error came from '{sv_best}' ({best_val:.3f} % against {base_val:.3f} % for the reference "
          f"configuration). The difference is {abs(base_val - best_val) / base_val * 100:.1f} % of the reference error; we kept the "
          f"pre-specified reference configuration so that no setting was tuned after the original submission.")
s_abl = (f"The parameter-matched one-layer model (85 units, {int(matched.params):,} parameters) reached a test nRMSE of "
         f"{matched.test:.3f} % against {float(sg.loc[BASE_CFG.name].test):.3f} % for the two-layer reference, so "
         + ("depth helps even when the number of parameters is held fixed." if matched.test > float(sg.loc[BASE_CFG.name].test)
            else "the benefit of the second layer disappears once the number of parameters is held fixed."))
s_overfit = (f"At the selected epoch the validation loss was on average {gap_ratio:.2f} times the training loss, and the "
             f"coefficient of variation of the test RMSE across seeds was {seed_cv:.1f} %.")
s_arima = (f"Order selection by AIC gave {orders}. The Ljung–Box test found remaining residual autocorrelation (5 % level) for "
           f"{words(lb_fail)} series. The rolling ARIMA forecast had a lower RMSE than persistence for {words(n_arima_vs_p_better)} of "
           f"the {words(N)} series, which is expected because daily price changes are close to uncorrelated.")
s_groups = (f"The mean Theil U2 was {core_u2:.3f} for the five technology stocks and {ext_u2:.3f} for the seven additional series."
            if ext else "")

TOK = {
    "N_SERIES": str(N), "N_SERIES_W": words(N), "N_SEEDS": str(len(SEEDS_MAIN)), "N_SEEDS_WF": str(len(SEEDS_WF)),
    "N_FOLDS": str(N_WF_FOLDS), "COST_BPS": str(COST_BPS), "PATIENCE": str(PATIENCE), "MAX_EPOCHS": str(MAX_EPOCHS),
    "DL_DATE": DL_INFO["download_date_utc"][:10], "YF_VER": DL_INFO["yfinance_version"],
    "R2_MIN": f"{r2_min:.3f}", "MAPE_MIN": f"{mape_min:.2f}", "MAPE_MAX": f"{mape_max:.2f}",
    "MASE_MIN": f"{mase_min:.3f}", "MASE_MAX": f"{mase_max:.3f}", "U2_MIN": f"{u2_min:.3f}", "U2_MAX": f"{u2_max:.3f}",
    "N_U2_BETTER": words(n_u2_better), "N_SIG_BETTER": words(n_sig_better), "N_SIG_WORSE": words(n_sig_worse),
    "N_SIG_BETTER_ARIMA": words(n_sig_better_arima), "N_SIG_WORSE_ARIMA": words(n_sig_worse_arima),
    "DA_MEAN": f"{da_mean:.1f}", "DA_MIN": f"{da_min:.1f}", "DA_MAX": f"{da_max:.1f}", "N_PT_SIG": words(n_pt_sig),
    "R2OOS_MEAN": f"{r2oos_mean:.2f}", "UP_MEAN": f"{up_mean:.0f}", "BEST_MODEL_COUNT": best_model_text,
    "WF_CELLS": str(wf_cells), "WF_BETTER": str(wf_better), "WF_PCT": f"{wf_better / wf_cells * 100:.0f}",
    "N_TRADE_BEAT": words(n_trade_beat), "GROSS_MINUS_NET": f"{gross_minus_net:.1f}", "LB_BEST": lb_best_txt,
    "SEED_CV": f"{seed_cv:.1f}", "GAP_RATIO": f"{gap_ratio:.2f}", "LB_FAIL": words(lb_fail),
    "S_DM": s_dm, "S_INFER": s_infer, "S_INFER_PLAIN": s_infer_plain, "S_DA": s_da + s_up, "S_TRADE": s_trade, "S_WF": s_wf,
    "S_VOL": s_vol, "S_SENS": s_sens, "S_ABL": s_abl, "S_OVERFIT": s_overfit, "S_ARIMA": s_arima, "S_GROUPS": s_groups,
    "S_PERSIST_MASE": (f"Persistence itself reached MASE values between {pers_mase.min():.3f} and {pers_mase.max():.3f}; "
                       f"the lowest MASE per series was obtained by {best_model_text}."),
}

# ---- additional data-driven sentences (keep every claim in the text tied to the numbers) ----
gap_list, move_list, bias_list, ci50 = [], [], {}, 0
for t in TL:
    y, idx, yt, yp, sc = test_arrays(t); lp = lstm_mean_pred(t)
    gap_list.append(np.mean(np.abs(lp - yp) / yp) * 100); move_list.append(np.mean(np.abs(yt - yp) / yp) * 100)
    bias_list[t] = float(np.mean((yt - lp) / yt) * 100)
    r = yt / yp - 1; rh = lp / yp - 1; m = r != 0
    lo_, hi2 = wilson_ci(int(np.sum(np.sign(r[m]) == np.sign(rh[m]))), int(m.sum())); ci50 += int(lo_ <= 0.5 <= hi2)
gap_m, move_m = float(np.mean(gap_list)), float(np.mean(move_list))
s_zoom = (f"On average the network's forecast differed from the previous day's price by {gap_m:.2f} % of the price, whereas the "
          f"actual price moved by {move_m:.2f} % per day, so the forecast mostly stays close to the last observed value.")
if gap_m >= 0.5 * move_m:
    s_zoom = (f"On average the network's forecast differed from the previous day's price by {gap_m:.2f} % of the price, compared "
              f"with an average actual daily move of {move_m:.2f} %.")
n_under = sum(v > 0 for v in bias_list.values()); tb = max(bias_list, key=lambda k: abs(bias_list[k]))
s_bias = (f"The network's mean error was positive, that is, it under-predicted on average, for {words(n_under)} of the {words(N)} series; "
          f"the largest mean bias was {bias_list[tb]:+.2f} % of the price ({tb}).")
pv_h = float(vg_h.nRMSE_pers.mean()); pv_n = float(vg_n.nRMSE_pers.mean())
s_volratio = f"For persistence, nRMSE on high-volatility days was on average {pv_h / pv_n:.1f} times its value on normal days."
sens_vals = sg.loc[[c.name for c in SENS_CFGS]].val
n_below = int((sens_vals < pers_val).sum())
s_sens2 = (f"Across all settings the mean validation nRMSE ranged from {sens_vals.min():.3f} % to {sens_vals.max():.3f} %, against "
           f"{pers_val:.3f} % for persistence; {words(n_below)} of the {len(sens_vals)} settings had a lower validation error than persistence.")
best_ep = float(np.mean([MAIN[(t, s_)]["meta"]["best_epoch"] for t in TL for s_ in SEEDS_MAIN]))
if gain:
    s_disc = ("Under the benchmark, significance and return analyses the network keeps a gain, but a much smaller one than the "
              "price-level scores suggest.")
    s_pract = ("the results suggest that the network's forecasts carry some information, but the gain is small and should be "
               "weighed against model risk and trading costs before it is used in any decision.")
else:
    s_disc = ("The benchmark, significance and return analyses show how little such scores reveal: once the random walk is used "
              "as the reference, most of the apparent skill disappears.")
    s_pract = ("the results suggest that a univariate recurrent network is not, on its own, a trading signal; its value, if any, "
               "is more likely to come from combining it with other information or from other tasks such as volatility or risk forecasting.")
TOK.update({
    "S_ZOOM": s_zoom, "N_CI50": words(ci50).capitalize(), "S_BIAS": s_bias, "S_VOLRATIO": s_volratio, "S_SENS2": s_sens2,
    "BEST_EP": f"{best_ep:.0f}", "S_DISC": s_disc, "S_PRACT": s_pract, "ARIMA_EVERY": str(ARIMA_REFIT_EVERY), "REPO_URL": REPO_URL,
    "S_DA_ABS": s_da,
    "S_WF_ABS": (f"In the walk-forward test the network's root mean squared error was below that of the random walk in {wf_better} of "
                 f"{wf_cells} series-fold combinations ({wf_better / wf_cells * 100:.0f} %)."),
    "S_TRADE_ABS": (f"After transaction costs, a simple long/flat trading rule based on the network's forecasts had a higher Sharpe ratio "
                    f"than buy-and-hold for {words(n_trade_beat)} of the {words(N)} series."),
})
json.dump(TOK, open(TAB_DIR / "text_values.json", "w"), indent=2, ensure_ascii=False)
for k_, v in TOK.items():
    if k_.startswith("S_"): print(f"{k_}: {v}\n")


In [ ]:
# ============================== 14. Fill the Word templates ==============================
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

TEMPLATES = {"SIGMA_Stock_Revised_TEMPLATE.docx": "SIGMA_Stock_Revised_FINAL_clean.docx",
             "SIGMA_Stock_Revised_TEMPLATE_highlighted.docx": "SIGMA_Stock_Revised_FINAL_highlighted.docx",
             "SIGMA_Stock_Response_TEMPLATE.docx": "SIGMA_Stock_Response_to_Reviewers_FINAL.docx"}

def find_template(name):
    for p in (Path(name), DOC_DIR / name, BASE / name, Path("/content") / name):
        if p.exists(): return p
    return None

missing = [n for n in TEMPLATES if find_template(n) is None]
if missing and IN_COLAB:
    from google.colab import files
    print("Please upload:", missing); up = files.upload()
    for k_ in up: (DOC_DIR / k_).write_bytes(up[k_])

def _set_cell_border(cell, **kw):
    tcPr = cell._tc.get_or_add_tcPr(); b = tcPr.find(qn("w:tcBorders"))
    if b is None: b = OxmlElement("w:tcBorders"); tcPr.append(b)
    for edge, val in kw.items():
        e = OxmlElement(f"w:{edge}"); e.set(qn("w:val"), val); e.set(qn("w:sz"), "8"); e.set(qn("w:color"), "000000"); b.append(e)

def insert_table(anchor_par, header, rows, font=7.5):
    doc = anchor_par.part.document
    tbl = doc.add_table(rows=1 + len(rows), cols=len(header))
    tbl.alignment = 1
    for j, h in enumerate(header):
        c = tbl.rows[0].cells[j]; c.text = ""; r = c.paragraphs[0].add_run(h); r.bold = True; r.font.size = Pt(font)
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        _set_cell_border(c, top="single", bottom="single")
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            c = tbl.rows[i + 1].cells[j]; c.text = ""; r = c.paragraphs[0].add_run(str(v)); r.font.size = Pt(font)
            c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER
            if i == len(rows) - 1: _set_cell_border(c, bottom="single")
    for row in tbl.rows:
        for c in row.cells:
            for p in c.paragraphs:
                p.paragraph_format.space_before = Pt(1); p.paragraph_format.space_after = Pt(1)
                for r in p.runs: r.font.name = "Times New Roman"
    anchor_par._p.addnext(tbl._tbl)
    anchor_par._p.getparent().remove(anchor_par._p)

def insert_figure(anchor_par, path, width_cm=16):
    for r in anchor_par.runs: r.text = ""
    anchor_par.alignment = WD_ALIGN_PARAGRAPH.CENTER
    anchor_par.add_run().add_picture(str(path), width=Cm(width_cm))

def _all_pars(doc):
    for p in doc.paragraphs: yield p
    for t in doc.tables:
        for row in t.rows:
            for c in row.cells:
                for p in c.paragraphs: yield p

def fill(src, dst):
    doc = Document(src)
    for p in list(doc.paragraphs):
        txt = p.text.strip()
        if txt.startswith("{{TABLE:") and txt.endswith("}}"):
            h, rows = TABLES[txt[8:-2]]; insert_table(p, h, rows)
        elif txt.startswith("{{FIG:") and txt.endswith("}}"):
            insert_figure(p, FIGS[txt[6:-2]])
    left = set()
    for p in _all_pars(doc):
        for r in p.runs:
            if "{{" in r.text:
                for k_, v in TOK.items():
                    r.text = r.text.replace("{{" + k_ + "}}", v)
                if "{{" in r.text: left.add(r.text)
    doc.save(dst)
    print("saved", dst, "| unfilled tokens:", left or "none")

for src_name, dst_name in TEMPLATES.items():
    src = find_template(src_name)
    if src is None:
        print("Template not found, skipped:", src_name); continue
    fill(src, DOC_DIR / dst_name)


In [ ]:
# ============================== 15. Package everything ==============================
import shutil
bundle = BASE / f"bundle_{MODE}"
if bundle.exists(): shutil.rmtree(bundle)
for d in (FIG_DIR, TAB_DIR, DOC_DIR, DATA_DIR):
    shutil.copytree(d, bundle / d.name)
zip_path = shutil.make_archive(str(BASE / f"SIGMA_Stock_revision_outputs_{MODE}"), "zip", root_dir=bundle)
print("Everything is in", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
